# Export data structures to .py files (run once)

In [2]:
import os
os.makedirs('src', exist_ok=True)
print(os.listdir('.'))

['02_avl_tree.ipynb', '03_min_heap.ipynb', '00_export_src.ipynb', '01_bst.ipynb', '.ipynb_checkpoints', '04_hash_table.ipynb', 'src']


In [3]:
%%writefile src/city_data.py
class CityData:
    def __init__(self, name, latitude, longitude, population, distance=0.0):
        self.name = name
        self.latitude = latitude
        self.longitude = longitude
        self.population = population
        self.distance = distance

    def __repr__(self):
        return f"City({self.name}, pop={self.population}, dist={self.distance})"

Writing src/city_data.py


In [4]:
%%writefile src/bst.py
from city_data import CityData


class BSTNode:
    def __init__(self, city: CityData):
        self.city = city
        self.left = None
        self.right = None


class BST:
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, city: CityData):
        self.root = self._insert(self.root, city)
        self.size += 1

    def _insert(self, node, city):
        if node is None:
            return BSTNode(city)
        if city.name < node.city.name:
            node.left = self._insert(node.left, city)
        elif city.name > node.city.name:
            node.right = self._insert(node.right, city)
        else:
            node.city = city
            self.size -= 1
        return node

    def search(self, name):
        return self._search(self.root, name)

    def _search(self, node, name):
        if node is None:
            return None
        if name == node.city.name:
            return node.city
        elif name < node.city.name:
            return self._search(node.left, name)
        else:
            return self._search(node.right, name)

    def delete(self, name):
        self.root, deleted = self._delete(self.root, name)
        if deleted:
            self.size -= 1
        return deleted

    def _delete(self, node, name):
        if node is None:
            return node, False

        if name < node.city.name:
            node.left, deleted = self._delete(node.left, name)
        elif name > node.city.name:
            node.right, deleted = self._delete(node.right, name)
        else:
            deleted = True
            if node.left is None and node.right is None:
                return None, deleted
            if node.left is None:
                return node.right, deleted
            if node.right is None:
                return node.left, deleted
            successor = self._min_node(node.right)
            node.city = successor.city
            node.right, _ = self._delete(node.right, successor.city.name)

        return node, deleted

    def _min_node(self, node):
        while node.left is not None:
            node = node.left
        return node

    def inorder(self):
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        if node:
            self._inorder(node.left, result)
            result.append(node.city)
            self._inorder(node.right, result)

    def height(self, node="root"):
        if node == "root":
            node = self.root
        if node is None:
            return -1
        return 1 + max(self.height(node.left), self.height(node.right))

Writing src/bst.py


In [5]:
%%writefile src/avl_tree.py
from city_data import CityData


class AVLNode:
    def __init__(self, city: CityData):
        self.city = city
        self.left = None
        self.right = None
        self.height = 1


class AVLTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def _height(self, node):
        return node.height if node else 0

    def _balance_factor(self, node):
        return self._height(node.left) - self._height(node.right) if node else 0

    def _update_height(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _rotate_right(self, y):
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        self._update_height(y)
        self._update_height(x)
        return x

    def _rotate_left(self, x):
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        self._update_height(x)
        self._update_height(y)
        return y

    def _rebalance(self, node):
        self._update_height(node)
        balance = self._balance_factor(node)

        if balance > 1:
            if self._balance_factor(node.left) < 0:
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)

        if balance < -1:
            if self._balance_factor(node.right) > 0:
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)

        return node

    def insert(self, city: CityData):
        self.root = self._insert(self.root, city)
        self.size += 1

    def _insert(self, node, city):
        if node is None:
            return AVLNode(city)
        if city.name < node.city.name:
            node.left = self._insert(node.left, city)
        elif city.name > node.city.name:
            node.right = self._insert(node.right, city)
        else:
            node.city = city
            self.size -= 1
            return node
        return self._rebalance(node)

    def search(self, name):
        return self._search(self.root, name)

    def _search(self, node, name):
        if node is None:
            return None
        if name == node.city.name:
            return node.city
        elif name < node.city.name:
            return self._search(node.left, name)
        else:
            return self._search(node.right, name)

    def delete(self, name):
        self.root, deleted = self._delete(self.root, name)
        if deleted:
            self.size -= 1
        return deleted

    def _delete(self, node, name):
        if node is None:
            return node, False

        if name < node.city.name:
            node.left, deleted = self._delete(node.left, name)
        elif name > node.city.name:
            node.right, deleted = self._delete(node.right, name)
        else:
            deleted = True
            if node.left is None and node.right is None:
                return None, deleted
            if node.left is None:
                return node.right, deleted
            if node.right is None:
                return node.left, deleted
            successor = self._min_node(node.right)
            node.city = successor.city
            node.right, _ = self._delete(node.right, successor.city.name)

        return self._rebalance(node), deleted

    def _min_node(self, node):
        while node.left is not None:
            node = node.left
        return node

    def inorder(self):
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        if node:
            self._inorder(node.left, result)
            result.append(node.city)
            self._inorder(node.right, result)

    def height(self):
        return self._height(self.root) - 1 if self.root else -1

Writing src/avl_tree.py


In [6]:
%%writefile src/min_heap.py
from city_data import CityData


class MinHeap:
    def __init__(self):
        self.heap = []

    def __len__(self):
        return len(self.heap)

    def is_empty(self):
        return len(self.heap) == 0

    def _parent(self, i):
        return (i - 1) // 2

    def _left(self, i):
        return 2 * i + 1

    def _right(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]

    def insert(self, city: CityData):
        self.heap.append(city)
        self._sift_up(len(self.heap) - 1)

    def _sift_up(self, i):
        while i > 0 and self.heap[i].distance < self.heap[self._parent(i)].distance:
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def peek_min(self):
        return self.heap[0] if self.heap else None

    def extract_min(self):
        if not self.heap:
            return None
        min_city = self.heap[0]
        last = self.heap.pop()
        if self.heap:
            self.heap[0] = last
            self._sift_down(0)
        return min_city

    def _sift_down(self, i):
        n = len(self.heap)
        while True:
            smallest = i
            left, right = self._left(i), self._right(i)
            if left < n and self.heap[left].distance < self.heap[smallest].distance:
                smallest = left
            if right < n and self.heap[right].distance < self.heap[smallest].distance:
                smallest = right
            if smallest == i:
                break
            self._swap(i, smallest)
            i = smallest

    @classmethod
    def heapify(cls, cities):
        h = cls()
        h.heap = list(cities)
        n = len(h.heap)
        for i in range(n // 2 - 1, -1, -1):
            h._sift_down(i)
        return h

Writing src/min_heap.py


In [7]:
%%writefile src/hash_table.py
from city_data import CityData


class HashTable:
    def __init__(self, capacity=8, load_factor_threshold=0.75):
        self.capacity = capacity
        self.load_factor_threshold = load_factor_threshold
        self.size = 0
        self.buckets = [[] for _ in range(self.capacity)]

    def _hash(self, key):
        return hash(key) % self.capacity

    def _load_factor(self):
        return self.size / self.capacity

    def insert(self, city: CityData):
        if self._load_factor() >= self.load_factor_threshold:
            self._resize()

        idx = self._hash(city.name)
        bucket = self.buckets[idx]
        for i, (k, _) in enumerate(bucket):
            if k == city.name:
                bucket[i] = (city.name, city)
                return
        bucket.append((city.name, city))
        self.size += 1

    def search(self, name):
        idx = self._hash(name)
        for k, city in self.buckets[idx]:
            if k == name:
                return city
        return None

    def delete(self, name):
        idx = self._hash(name)
        bucket = self.buckets[idx]
        for i, (k, _) in enumerate(bucket):
            if k == name:
                del bucket[i]
                self.size -= 1
                return True
        return False

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:
            for _, city in bucket:
                self.insert(city)

    def bucket_lengths(self):
        return [len(b) for b in self.buckets]

Writing src/hash_table.py


In [8]:
import os
print(os.listdir('src'))

['min_heap.py', 'avl_tree.py', 'hash_table.py', 'bst.py', 'city_data.py']
